# Convert LeHaMoC comoving SED to observed flux

This notebook reads the photon spectrum saved by `LeHaMoC.py` in the comoving frame and converts it to observed `νFν` using the Doppler factor and redshift. Optional EBL absorption can be included through `ebltable`. If no EBL absorption is included ebl_model should be set to None `ebl_model = None`

In [ ]:
!python LeHaMoC.py

Progress...:  80%|████████████████████████▊      | 8/10 [00:08<00:02,  1.07s/it]

In [ ]:
import numpy as np
import astropy.units as u
import pandas as pd
import matplotlib.pyplot as plt
from astropy import constants as const
from astropy.cosmology import Planck18 as cosmo


#######################
#constants# 
#######################
G = (const.G).cgs.value       
c = (const.c).cgs.value     
Ro = (const.R_sun).cgs.value            
Mo = (const.M_sun).cgs.value       
yr = (u.yr).to(u.s)                
kpc = (u.kpc).to(u.cm)             
pc = (u.pc).to(u.cm)              
m_pr = (u.M_p).to(u.g)         
m_el = (u.M_e).to(u.g)         
kb = (const.k_B).cgs.value
h = (const.h).cgs.value 
q = (const.e.gauss).value                
sigmaT = (const.sigma_T).cgs.value               
eV = (u.eV).to(u.erg)   
B_cr = 2*np.pi*m_el**2*c**3/(h*q)

# Read the Parameters.txt file
fileName = "Parameters.txt"
fileObj = open(fileName)
params = {}
for line in fileObj:
    line=line.strip()
    key_value = line.split("=")
    params[key_value[0].strip()] = float(key_value[1].strip())
    
time_init = float(params['time_init']) #R0/c
time_end = float(params['time_end']) #R0/c
step_alg = float(params['step_alg']) #R0/c
grid_nu = int(float(params['grid_nu']))    
grid_g_el = int(float(params['grid_g_el']))
grid_g_pr = int(float(params['grid_g_pr']))

R0 = 10**float(params['R0']) #log
step_alg = float(params['step_alg']) #R0/c
M_F = float(params['B0']) 
delta = float(params["delta"])
# Read Outpuf files
# LeHaMoC code results
El_dis_lemoc_t3 = pd.read_csv("Pairs_Distribution.txt", names=["logx","logy"], sep=" ",  skiprows=0) 
Ph_dis_lemoc_t3 = pd.read_csv("Photons_Distribution.txt", names=["logx","logy"], sep=" ",  skiprows=0) 

nu_tot = np.array(10**Ph_dis_lemoc_t3["logx"])
Spec_temp_tot = np.array(10**Ph_dis_lemoc_t3["logy"])
g_el  = np.array(10**El_dis_lemoc_t3["logx"])
N_el  = np.array(10**El_dis_lemoc_t3["logy"])


def leha_sed_to_observer(nu_prime_hz, nuLnu_prime_erg_s, z, delta_D, ebl_model=None, ebl_scale=1.0, cosmology=cosmo,):
    """
    Convert LeHaMoC comoving SED to observed nuFnu, optionally with EBL absorption.

    Parameters
    ----------
    nu_prime_hz : Comoving frequency grid [Hz].
    nuLnu_prime_erg_s : Comoving SED nu' L'_{nu'} [erg/s].
    z : Source redshift.
    delta_D : Doppler factor.
    ebl_model : Example: 'finke2022', 'saldana-lopez', 'dominguez', 'franceschini2017'. If None, no EBL attenuation is applied.
    ebl_scale : Multiplier alpha for tau_EBL. Usually alpha=1.
    cosmology : Cosmology used to compute luminosity distance.

    Returns
    -------
    nu_obs_hz : Observed frequency [Hz].
    nuFnu_obs : Observed attenuated SED [erg cm^-2 s^-1].
    nuFnu_int : Observed intrinsic SED before EBL [erg cm^-2 s^-1].
    tau_ebl : EBL optical depth.
    """

    nu_prime_hz = np.asarray(nu_prime_hz, dtype=float)
    nuLnu_prime_erg_s = np.asarray(nuLnu_prime_erg_s, dtype=float)

    # Frequency transformation
    nu_obs_hz = delta_D * nu_prime_hz / (1. + z)

    # Luminosity distance
    dL_cm = cosmology.luminosity_distance(z).to_value("cm")

    # Intrinsic observed SED, before EBL
    nuFnu_int = delta_D**4 * nuLnu_prime_erg_s / (4. * np.pi * dL_cm**2)

    # Default: no EBL
    tau_ebl = np.zeros_like(nu_obs_hz)

    if ebl_model is not None:
        from ebltable.tau_from_model import OptDepth

        # ebltable expects observed energy in TeV
        E_obs_TeV = (h * nu_obs_hz * u.erg).to_value(u.TeV)

        tau_model = OptDepth.readmodel(ebl_model)

        # Avoid asking the table for radio/optical/X-ray frequencies.
        # EBL attenuation matters only for gamma rays.
        gamma_mask = E_obs_TeV > 1e-6  # 1 MeV in TeV units; safely low

        tau_ebl[gamma_mask] = np.squeeze(tau_model.opt_depth(z, E_obs_TeV[gamma_mask]))
        tau_ebl = np.maximum(tau_ebl, 0.0)
    attenuation = np.exp(-ebl_scale * tau_ebl)
    nuFnu_obs = nuFnu_int * attenuation
    return nu_obs_hz, nuFnu_obs, nuFnu_int, tau_ebl
#delta =  You can define delta also here.
redshift = 0.186
LIST_ALL = leha_sed_to_observer(nu_tot[-grid_nu:], Spec_temp_tot[-grid_nu:], redshift, 24.2, ebl_model="finke2022", ebl_scale=1.0, cosmology=cosmo,)
plt.loglog(LIST_ALL[0], LIST_ALL[1],label="Finke et al. 2022 A", c = "k") 
plt.ylabel("ν F$_ν$ [erg s$^{-1}$]")
plt.xlabel("ν[Hz]")

plt.ylim(10**(-19.),10**(-10.))
plt.xlim(10**(10.),10**(33.))
plt.legend(fontsize=10)